# Workflow Ekonometrii Panelowej — Python/linearmodels

## Cel projektu
Analiza determinant bezrobocia w województwach Polski w latach 2014–2024 z wykorzystaniem panelowych modeli efektów stałych (FE) oraz testów diagnostycznych.

## Struktura workflow
1. **Inicjalizacja danych** — wczytanie, konwersja, przygotowanie panelu
2. **Korelacje pooled** — wstępna ocena siły związków
3. **Modelowanie** — Pooled OLS, FE, FE TwoWay, Between, RE
4. **Testowanie specyfikacji** — Hausman (FE vs RE), testy LM
5. **Diagnostyka reszt** — autokorelacja, heteroskedastyczność, korelacja przekrojowa
6. **Odporna inferencja** — błędy standardowe z klastrowaniem
7. **Wnioski i zalecenia**

## Pakiety
- `pandas`, `numpy` — operacje na danych
- `linearmodels` — modele panelowe FE, RE, pooled, between
- `statsmodels`, `scipy` — testy diagnostyczne i statystyczne

In [109]:
import pandas as pd
import numpy as np

import linearmodels as lm

In [110]:
df = pd.read_csv("data/processed/merged_data.csv")

In [111]:
df.columns.tolist()

['region',
 'wynagrodzenie',
 'rok',
 'bezrobotni_w_liczbie_ludności_w_wieku_produkcyjnym',
 'ogółem',
 'gimnazjalne,_podstawowe_i_niższe_ogółem_wartość_liczbowa',
 'gimnazjalne,_podstawowe_i_niższe_ogółem_wskaźnik_precyzji',
 'policealne_oraz_średnie_zawodowe/branżowe_ogółem_wartość_liczbowa',
 'policealne_oraz_średnie_zawodowe/branżowe_ogółem_wskaźnik_precyzji',
 'wyższe_ogółem_wartość_liczbowa',
 'wyższe_ogółem_wskaźnik_precyzji',
 'zasadnicze_zawodowe/branżowe_ogółem_wartość_liczbowa',
 'zasadnicze_zawodowe/branżowe_ogółem_wskaźnik_precyzji',
 'średnie_(łącznie_z_zasadniczym_zawodowym/branżowym_i_policealnym)_ogółem_wartość_liczbowa',
 'średnie_(łącznie_ze_średnim_zawodowym/branżowym_i_ogólnokształcącym)_ogółem_wartość_liczbowa',
 'średnie_ogólnokształcące_ogółem_wartość_liczbowa',
 'średnie_ogólnokształcące_ogółem_wskaźnik_precyzji',
 'średnie_zawodowe/branżowe_ogółem_wartość_liczbowa',
 'średnie_zawodowe/branżowe_ogółem_wskaźnik_precyzji',
 'saldo_migracji_ogółem',
 'wymeldowania

In [112]:
df.columns = df.columns.str.replace(' ', '_')

In [113]:
df.columns.tolist()

['region',
 'wynagrodzenie',
 'rok',
 'bezrobotni_w_liczbie_ludności_w_wieku_produkcyjnym',
 'ogółem',
 'gimnazjalne,_podstawowe_i_niższe_ogółem_wartość_liczbowa',
 'gimnazjalne,_podstawowe_i_niższe_ogółem_wskaźnik_precyzji',
 'policealne_oraz_średnie_zawodowe/branżowe_ogółem_wartość_liczbowa',
 'policealne_oraz_średnie_zawodowe/branżowe_ogółem_wskaźnik_precyzji',
 'wyższe_ogółem_wartość_liczbowa',
 'wyższe_ogółem_wskaźnik_precyzji',
 'zasadnicze_zawodowe/branżowe_ogółem_wartość_liczbowa',
 'zasadnicze_zawodowe/branżowe_ogółem_wskaźnik_precyzji',
 'średnie_(łącznie_z_zasadniczym_zawodowym/branżowym_i_policealnym)_ogółem_wartość_liczbowa',
 'średnie_(łącznie_ze_średnim_zawodowym/branżowym_i_ogólnokształcącym)_ogółem_wartość_liczbowa',
 'średnie_ogólnokształcące_ogółem_wartość_liczbowa',
 'średnie_ogólnokształcące_ogółem_wskaźnik_precyzji',
 'średnie_zawodowe/branżowe_ogółem_wartość_liczbowa',
 'średnie_zawodowe/branżowe_ogółem_wskaźnik_precyzji',
 'saldo_migracji_ogółem',
 'wymeldowania

In [114]:
"""
MODEL PANELOWY — Fixed Effects (odpowiednik R::plm)
====================================================
Wymaga wcześniejszego przejścia przez audyt_danych_panelowych.py

Instalacja:
    pip install pandas numpy linearmodels statsmodels scipy

Uruchomienie:
    python model_panelowy.py
"""

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from linearmodels.panel import PanelOLS, PooledOLS, BetweenOLS, RandomEffects
from linearmodels.panel import compare
from scipy import stats
import statsmodels.formula.api as smf

# ─────────────────────────────────────────────
# KONFIGURACJA — dostosuj do swojego pliku
# ─────────────────────────────────────────────
PLIK_DANYCH   = "data/processed/merged_data.csv"
SEP           = ","

KOLUMNA_ID    = "region"
KOLUMNA_CZAS  = "rok"

# Zmienna zależna
ZMIENNA_Y     = "bezrobotni_w_liczbie_ludności_w_wieku_produkcyjnym"

# Kluczowa determinanta (wybierz JEDNĄ zgodnie z sugestią opiekuna)
# Opcja A: inwestycje
# Opcja B: gęstość podmiotów
ZMIENNA_KLUCZ = "inwestycje_zl"
# ZMIENNA_KLUCZ = "podmiot_nowo_zarejestr_na_10_tys_ludnosci_w_wieku_produkcyjnym"

# Zmienne kontrolne — wpisz te, które przeżyły audyt korelacji
ZMIENNE_KONTROLNE = [
    "wynagrodzenie",
    "saldo_migracji_ogółem",
    "liczba_pomiotow_gospodarczych",
    "mieszkania oddane do użytkowania na 10 tys. ludności",
    # dodaj/usuń wg wyników korelacji z audytu
]
# ─────────────────────────────────────────────


In [115]:
# ══════════════════════════════════════════════
# 0. WCZYTANIE I PRZYGOTOWANIE DANYCH
# ══════════════════════════════════════════════
print("=" * 65)
print("KROK 0 — Wczytanie i przygotowanie danych panelowych")
print("=" * 65)

if PLIK_DANYCH.endswith(".xlsx"):
    df = pd.read_excel(PLIK_DANYCH)
else:
    df = pd.read_csv(PLIK_DANYCH, sep=SEP, low_memory=False)

df.columns = df.columns.str.replace(' ', '_')

# Konwersja kolumn numerycznych (polskie dane mogą mieć przecinki)
for col in [ZMIENNA_Y, ZMIENNA_KLUCZ] + ZMIENNE_KONTROLNE:
    if col in df.columns:
        df[col] = (df[col].astype(str)
                          .str.replace(",", ".", regex=False)
                          .str.replace(" ", "", regex=False))
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Rok jako integer
df[KOLUMNA_CZAS] = pd.to_numeric(df[KOLUMNA_CZAS], errors="coerce").astype("Int64")

# Usuwamy wiersze z NA w kluczowych kolumnach
cols_modelu = [KOLUMNA_ID, KOLUMNA_CZAS, ZMIENNA_Y, ZMIENNA_KLUCZ] + ZMIENNE_KONTROLNE
cols_dostepne = [c for c in cols_modelu if c in df.columns]
df_model = df[cols_dostepne].dropna()

print(f"  Obserwacje po usunięciu NA: {len(df_model):,}")
print(f"  Regiony: {df_model[KOLUMNA_ID].nunique()}")
print(f"  Lata: {sorted(df_model[KOLUMNA_CZAS].unique())}\n")


KROK 0 — Wczytanie i przygotowanie danych panelowych
  Obserwacje po usunięciu NA: 176
  Regiony: 16
  Lata: [np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]



In [116]:
# ══════════════════════════════════════════════
# 1. KORELACJE POOLED (wskazówka opiekuna)
# ══════════════════════════════════════════════
print("=" * 65)
print("KROK 1 — Korelacje pooled (wszystkie obs. razem, bez podziału na panele)")
print("=" * 65)
print("  Opiekun: 'jeżeli korelacja będzie niska, trudno zbudować dobry model'\n")

y_series = df_model[ZMIENNA_Y]
zmienne_do_korelacji = [ZMIENNA_KLUCZ] + [c for c in ZMIENNE_KONTROLNE if c in df_model.columns]

wyniki_korelacji = []
for zmienna in zmienne_do_korelacji:
    x = df_model[zmienna]
    mask = x.notna() & y_series.notna()
    if mask.sum() < 10:
        continue
    r, p = stats.pearsonr(x[mask], y_series[mask])
    wyniki_korelacji.append({
        "zmienna": zmienna,
        "r_pearson": round(r, 4),
        "p_value": round(p, 4),
        "istotna": "✔" if p < 0.05 else "✖",
        "ocena": ("✔✔ silna" if abs(r) >= 0.5 else
                  "✔ umiark." if abs(r) >= 0.3 else
                  "~ słaba" if abs(r) >= 0.1 else
                  "✖ brak")
    })

df_corr = pd.DataFrame(wyniki_korelacji).sort_values("r_pearson", key=abs, ascending=False)
print(df_corr.to_string(index=False))

r_klucz = df_corr.loc[df_corr["zmienna"] == ZMIENNA_KLUCZ, "r_pearson"]
if not r_klucz.empty:
    r_val = r_klucz.values[0]
    if abs(r_val) < 0.15:
        print(f"\n  ⚠ Korelacja kluczowej zmiennej z Y wynosi {r_val:.3f} — niska!")
        print("    Rozważ: inną zmienną kluczową, transformację log(), lub uzasadnienie merytoryczne.")
    else:
        print(f"\n  ✔ Korelacja kluczowej zmiennej z Y: {r_val:.3f}")
print()


KROK 1 — Korelacje pooled (wszystkie obs. razem, bez podziału na panele)
  Opiekun: 'jeżeli korelacja będzie niska, trudno zbudować dobry model'

                      zmienna  r_pearson  p_value istotna     ocena
liczba_pomiotow_gospodarczych    -0.6383   0.0000       ✔  ✔✔ silna
                wynagrodzenie    -0.5246   0.0000       ✔  ✔✔ silna
                inwestycje_zl    -0.4937   0.0000       ✔ ✔ umiark.
        saldo_migracji_ogółem    -0.2665   0.0004       ✔   ~ słaba

  ✔ Korelacja kluczowej zmiennej z Y: -0.494



In [117]:
# ══════════════════════════════════════════════
# 2. DEKLARACJA DANYCH PANELOWYCH
#    odpowiednik: pdata.frame(df, index=c("region","rok"))
# ══════════════════════════════════════════════
print("=" * 65)
print("KROK 2 — Deklaracja struktury panelowej (indeks: region × rok)")
print("=" * 65)

# linearmodels wymaga MultiIndex: (entity, time)
df_panel = df_model.copy()
df_panel = df_panel.set_index([KOLUMNA_ID, KOLUMNA_CZAS])

# Formuła po prawej stronie
rhs_zmienne = [ZMIENNA_KLUCZ] + [c for c in ZMIENNE_KONTROLNE if c in df_panel.columns]
formula_rhs = " + ".join(rhs_zmienne)
print(f"  Y  = {ZMIENNA_Y}")
print(f"  X  = {formula_rhs}\n")


KROK 2 — Deklaracja struktury panelowej (indeks: region × rok)
  Y  = bezrobotni_w_liczbie_ludności_w_wieku_produkcyjnym
  X  = inwestycje_zl + wynagrodzenie + saldo_migracji_ogółem + liczba_pomiotow_gospodarczych



In [118]:
# ══════════════════════════════════════════════
# 3. MODEL POOLED OLS (punkt odniesienia)
#    dane traktowane jak zwykła regresja, bez panelu
# ══════════════════════════════════════════════
print("=" * 65)
print("KROK 3 — Model Pooled OLS (punkt odniesienia, ignoruje strukturę panelową)")
print("=" * 65)

model_pooled = PooledOLS.from_formula(
    f"{ZMIENNA_Y} ~ 1 + {formula_rhs}",
    data=df_panel
)
wynik_pooled = model_pooled.fit(cov_type="clustered", cluster_entity=True)
print(wynik_pooled.summary.tables[1])
print(f"  R²: {wynik_pooled.rsquared:.4f}\n")


KROK 3 — Model Pooled OLS (punkt odniesienia, ignoruje strukturę panelową)
                                       Parameter Estimates                                       
                               Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
-------------------------------------------------------------------------------------------------
Intercept                         5.5323     0.9089     6.0868     0.0000      3.7382      7.3264
inwestycje_zl                  5.558e-05  6.845e-05     0.8119     0.4180  -7.954e-05      0.0002
wynagrodzenie                 -2.322e-05     0.0001    -0.1553     0.8768     -0.0003      0.0003
saldo_migracji_ogółem          6.576e-05  5.659e-05     1.1620     0.2469  -4.595e-05      0.0002
liczba_pomiotow_gospodarczych    -0.0020     0.0009    -2.3067     0.0223     -0.0037     -0.0003
  R²: 0.4747



In [119]:
# ══════════════════════════════════════════════
# 4. MODEL FE — WITHIN (efekty indywidualne)
#    odpowiednik: plm(..., model="within", effect="individual")
# ══════════════════════════════════════════════
print("=" * 65)
print("KROK 4 — Model FE Within (efekty indywidualne — różnice wewnątrz regionu)")
print("=" * 65)
print("  R: plm(Y ~ X, model='within', effect='individual')\n")

model_fe = PanelOLS.from_formula(
    f"{ZMIENNA_Y} ~ 1 + {formula_rhs} + EntityEffects",
    data=df_panel
)
wynik_fe = model_fe.fit(cov_type="clustered", cluster_entity=True)
print(wynik_fe.summary.tables[1])
print(f"  R² (within): {wynik_fe.rsquared_within:.4f}")
print(f"  R² (between): {wynik_fe.rsquared_between:.4f}")
print(f"  R² (overall): {wynik_fe.rsquared_overall:.4f}")



KROK 4 — Model FE Within (efekty indywidualne — różnice wewnątrz regionu)
  R: plm(Y ~ X, model='within', effect='individual')

                                       Parameter Estimates                                       
                               Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
-------------------------------------------------------------------------------------------------
Intercept                         5.7181     0.3349     17.073     0.0000      5.0566      6.3797
inwestycje_zl                   1.11e-05  3.379e-05     0.3283     0.7431  -5.566e-05   7.785e-05
wynagrodzenie                   5.08e-05   7.16e-05     0.7095     0.4791  -9.064e-05      0.0002
saldo_migracji_ogółem          3.834e-05  8.497e-05     0.4512     0.6525     -0.0001      0.0002
liczba_pomiotow_gospodarczych    -0.0021     0.0003    -6.6714     0.0000     -0.0028     -0.0015
  R² (within): 0.5075
  R² (between): 0.3461
  R² (overall): 0.4184


In [120]:
# ══════════════════════════════════════════════
# 5. MODEL FE — TWOWAY (efekty indywidualne + czasowe)
#    odpowiednik: plm(..., model="within", effect="twoways")
#    opiekun: "przewiduję znaczną istotność komponentu czasowego"
# ══════════════════════════════════════════════
print("=" * 65)
print("KROK 5 — Model FE TwoWay (efekty regionów + efekty lat)")
print("=" * 65)
print("  R: plm(Y ~ X + factor(rok), model='within', effect='twoways')")
print("  Opiekun: 'przewiduję znaczną istotność komponentu czasowego'\n")

model_fe_time = PanelOLS.from_formula(
    f"{ZMIENNA_Y} ~ 1 + {formula_rhs} + EntityEffects + TimeEffects",
    data=df_panel
)
wynik_fe_time = model_fe_time.fit(cov_type="clustered", cluster_entity=True)
print(wynik_fe_time.summary.tables[1])
print(f"  R² (within): {wynik_fe_time.rsquared_within:.4f}")
print(f"  R² (between): {wynik_fe_time.rsquared_between:.4f}")
print(f"  R² (overall): {wynik_fe_time.rsquared_overall:.4f}")


KROK 5 — Model FE TwoWay (efekty regionów + efekty lat)
  R: plm(Y ~ X + factor(rok), model='within', effect='twoways')
  Opiekun: 'przewiduję znaczną istotność komponentu czasowego'

                                       Parameter Estimates                                       
                               Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
-------------------------------------------------------------------------------------------------
Intercept                         0.4520     1.5567     0.2904     0.7719     -2.6245      3.5285
inwestycje_zl                 -1.447e-05  2.279e-05    -0.6351     0.5264   -5.95e-05   3.056e-05
wynagrodzenie                     0.0003     0.0003     1.0182     0.3103     -0.0003      0.0010
saldo_migracji_ogółem          4.512e-05  1.466e-05     3.0768     0.0025   1.614e-05    7.41e-05
liczba_pomiotow_gospodarczych -5.965e-05     0.0013    -0.0463     0.9631     -0.0026      0.0025
  R² (within): -1.6576
  R² (bet

In [121]:
# ══════════════════════════════════════════════
# 6. MODEL BETWEEN (różnice między regionami)
#    odpowiednik: plm(..., model="between")
#    opiekun: "within-between"
# ══════════════════════════════════════════════
print("=" * 65)
print("KROK 6 — Model Between (różnice między regionami, średnie w czasie)")
print("=" * 65)
print("  R: plm(Y ~ X, model='between')\n")

model_be = BetweenOLS.from_formula(
    f"{ZMIENNA_Y} ~ 1 + {formula_rhs}",
    data=df_panel
)
wynik_be = model_be.fit(cov_type="robust")
print(wynik_be.summary.tables[1])
print(f"  R²: {wynik_be.rsquared:.4f}\n")



KROK 6 — Model Between (różnice między regionami, średnie w czasie)
  R: plm(Y ~ X, model='between')

                                       Parameter Estimates                                       
                               Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
-------------------------------------------------------------------------------------------------
Intercept                         5.6991     3.7836     1.5063     0.1602     -2.6286      14.027
inwestycje_zl                     0.0001     0.0002     0.6581     0.5240     -0.0003      0.0005
wynagrodzenie                    -0.0001     0.0009    -0.1348     0.8952     -0.0021      0.0019
saldo_migracji_ogółem          5.646e-05  9.617e-05     0.5871     0.5690     -0.0002      0.0003
liczba_pomiotow_gospodarczych    -0.0021     0.0011    -1.7961     0.1000     -0.0046      0.0005
  R²: 0.4569



In [122]:
# ══════════════════════════════════════════════
# 7. TEST HAUSMANA — FE vs RE
#    czy Fixed Effects są lepsze niż Random Effects?
# ══════════════════════════════════════════════
print("=" * 65)
print("KROK 7 — Test Hausmana: Fixed Effects vs Random Effects")
print("=" * 65)
print("  H0: efekty indywidualne NIE są skorelowane ze zmiennymi X → RE OK")
print("  H1: efekty indywidualne SĄ skorelowane ze zmiennymi X → FE wymagane\n")

model_re = RandomEffects.from_formula(
    f"{ZMIENNA_Y} ~ 1 + {formula_rhs}",
    data=df_panel
)
wynik_re = model_re.fit(cov_type="robust")

# Ręczna statystyka Hausmana
b_fe = wynik_fe.params
b_re = wynik_re.params
common = b_fe.index.intersection(b_re.index)
diff = b_fe[common] - b_re[common]

V_fe = wynik_fe.cov.loc[common, common]
V_re = wynik_re.cov.loc[common, common]
V_diff = V_fe - V_re

try:
    V_diff_inv = np.linalg.pinv(V_diff.values)
    hausman_stat = float(diff.values @ V_diff_inv @ diff.values)
    hausman_df   = len(common)
    hausman_p    = 1 - stats.chi2.cdf(hausman_stat, df=hausman_df)
    print(f"  Statystyka χ²({hausman_df}) = {hausman_stat:.4f}")
    print(f"  p-value = {hausman_p:.4f}")
    if hausman_p < 0.05:
        print("  ➤ Odrzucamy H0 — stosuj Fixed Effects (FE) ✔")
    else:
        print("  ➤ Brak podstaw do odrzucenia H0 — Random Effects mogą być OK")
except Exception as e:
    print(f"  ⚠ Nie udało się policzyć testu Hausmana: {e}")
print()


KROK 7 — Test Hausmana: Fixed Effects vs Random Effects
  H0: efekty indywidualne NIE są skorelowane ze zmiennymi X → RE OK
  H1: efekty indywidualne SĄ skorelowane ze zmiennymi X → FE wymagane

  Statystyka χ²(5) = -0.3093
  p-value = 1.0000
  ➤ Brak podstaw do odrzucenia H0 — Random Effects mogą być OK



In [123]:

# ══════════════════════════════════════════════
# 8. PORÓWNANIE MODELI
# ══════════════════════════════════════════════
print("=" * 65)
print("KROK 8 — Porównanie modeli (Pooled / FE / FE TwoWay)")
print("=" * 65)

porownanie = compare({
    "Pooled OLS":  wynik_pooled,
    "FE (entity)": wynik_fe,
    "FE (twoway)": wynik_fe_time,
    "RE":          wynik_re,
}, stars=True)
print(porownanie)

KROK 8 — Porównanie modeli (Pooled / FE / FE TwoWay)
                                                                                                                       Model Comparison                                                                                                                      
                                                                              Pooled OLS                                            FE (entity)                                            FE (twoway)                                                     RE
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Dep. Variable                         bezrobotni_w_liczbie_ludności_w_wieku_produkcyjnym     bezrobotni_w_liczbie_ludności_w_wieku_produkcyjnym     bezrobotni_w_liczbie_ludności_w_wieku

In [124]:
# ══════════════════════════════════════════════
# 9. PODSUMOWANIE
# ══════════════════════════════════════════════
print("\n" + "=" * 65)
print("PODSUMOWANIE DO KONSULTACJI Z OPIEKUNEM")
print("=" * 65)
print(f"""
  Kluczowa determinanta : {ZMIENNA_KLUCZ}
  Zmienna zależna       : {ZMIENNA_Y}
  Liczba obserwacji     : {len(df_model):,}

  Model rekomendowany   : FE TwoWay (efekty regionów + lat)
  Uzasadnienie          : opiekun przewiduje istotny komponent czasowy
                          (historia bezrobocia w Polsce)

  Następne kroki:
    1. Sprawdź istotność zmiennych — usuń nieistotne i przetestuj ponownie
    2. Sprawdź, czy R²(within) wzrósł po dodaniu TimeEffects
    3. Na konsultacji pokaż wynik testu Hausmana i tabelę compare()
    4. Rozważ log(Y) jeśli bezrobocie ma wysoką skośność (audyt, krok 4)
""")


PODSUMOWANIE DO KONSULTACJI Z OPIEKUNEM

  Kluczowa determinanta : inwestycje_zl
  Zmienna zależna       : bezrobotni_w_liczbie_ludności_w_wieku_produkcyjnym
  Liczba obserwacji     : 176

  Model rekomendowany   : FE TwoWay (efekty regionów + lat)
  Uzasadnienie          : opiekun przewiduje istotny komponent czasowy
                          (historia bezrobocia w Polsce)

  Następne kroki:
    1. Sprawdź istotność zmiennych — usuń nieistotne i przetestuj ponownie
    2. Sprawdź, czy R²(within) wzrósł po dodaniu TimeEffects
    3. Na konsultacji pokaż wynik testu Hausmana i tabelę compare()
    4. Rozważ log(Y) jeśli bezrobocie ma wysoką skośność (audyt, krok 4)



In [133]:
# ══════════════════════════════════════════════
# 10. TESTY DIAGNOSTYCZNE — BATCH I
# ══════════════════════════════════════════════
print("\n" + "=" * 65)
print("KROK 10 — Testy LM (Mnożniki Lagrange'a) na modelu FE TwoWay")
print("=" * 65)
print("  H0 (każdy test): efekty nie są istotne → stay with Pooled")
print("  H1: efekty SĄ istotne → leave Pooled\n")

# Testy diagnostyczne wymagan ze statsmodels
from linearmodels.panel import FirstDifferenceOLS
import statsmodels.api as sm
from scipy.stats import chi2

# Pobierz rezydua z modelu FE TwoWay
residuala_fe = wynik_fe_time.resids

# Test autokorelacji Wooldridge'a (ważny dla paneli)
print("  Test Wooldridge'a (autokorelacja AR(1) w panelu):")
# Transformacja: czy rho odbiega od 0?
lag_resid = residuala_fe.groupby(level=0).shift(1)
mask = residuala_fe.notna() & lag_resid.notna()
rho_est = residuala_fe[mask].corr(lag_resid[mask])
print(f"    ρ̂ (korelacja lag1): {rho_est:.4f}")
if abs(rho_est) > 0.3:
    print(f"    ⚠ Potencjalna autokorelacja (|ρ| > 0.3)")
else:
    print(f"    ✔ Brak silnej autokorelacji")


KROK 10 — Testy LM (Mnożniki Lagrange'a) na modelu FE TwoWay
  H0 (każdy test): efekty nie są istotne → stay with Pooled
  H1: efekty SĄ istotne → leave Pooled

  Test Wooldridge'a (autokorelacja AR(1) w panelu):
    ρ̂ (korelacja lag1): 0.6617
    ⚠ Potencjalna autokorelacja (|ρ| > 0.3)



  Test Breuscha-Pagana (heteroskedastyczność):
    Test stat: 0.7722, p-value: 0.3795
    ✔ Brak istotnej heteroskedastyczności


In [135]:
# ══════════════════════════════════════════════
# 10. TESTY DIAGNOSTYCZNE — BATCH I
# ══════════════════════════════════════════════

print("\n" + "=" * 65)
print("KROK 10 — Testy diagnostyczne FE TwoWay")
print("=" * 65)

print("  H0 (każdy test): brak problemu (OK model)")
print("  H1: problem obecny (model wymaga korekt)\n")

import numpy as np
from statsmodels.stats.diagnostic import het_breuschpagan

# =========================
# REZYDUA
# =========================
resid = wynik_fe_time.resids

# =========================
# 1. AUTOKORELACJA (Wooldridge proxy)
# =========================
print("Test autokorelacji (AR(1) proxy):")

lag_resid = resid.groupby(level=0).shift(1)

mask = resid.notna() & lag_resid.notna()

rho_est = resid[mask].corr(lag_resid[mask])

print(f"  ρ̂ (lag 1): {rho_est:.4f}")

if abs(rho_est) > 0.3:
    print("  ⚠ możliwa autokorelacja")
else:
    print("  ✔ brak silnej autokorelacji")


# =========================
# 2. HETEROSKEDASTYCZNOŚĆ (Breusch–Pagan)
# =========================
# Test heteroskedastyczności — test Breuscha-Pagana na residuach
print("\n  Test Breuscha-Pagana (heteroskedastyczność):")
from statsmodels.stats.diagnostic import het_breuschpagan
import statsmodels.api as sm

# Pobierz fitted values jako 1D array
y_pred = wynik_fe_time.fitted_values.values.flatten()

# Zbuduj macierz exog: stała + fitted values (wymagane przez het_breuschpagan)
X_bp = sm.add_constant(y_pred)

# Upewnij się że residua są 1D array
resid_vals = residuala_fe.values.flatten()

# Usuń NaN (jeśli są)
mask_bp = ~np.isnan(resid_vals) & ~np.isnan(X_bp[:, 1])
bp_test = het_breuschpagan(resid_vals[mask_bp], X_bp[mask_bp])

print(f"    Test stat: {bp_test[0]:.4f}, p-value: {bp_test[1]:.4f}")
if bp_test[1] < 0.05:
    print("    ⚠ Heteroskedastyczność wykryta (p < 0.05)")
else:
    print("    ✔ Brak istotnej heteroskedastyczności")


print("\n" + "=" * 65)


KROK 10 — Testy diagnostyczne FE TwoWay
  H0 (każdy test): brak problemu (OK model)
  H1: problem obecny (model wymaga korekt)

Test autokorelacji (AR(1) proxy):
  ρ̂ (lag 1): 0.6617
  ⚠ możliwa autokorelacja

  Test Breuscha-Pagana (heteroskedastyczność):
    Test stat: 0.7722, p-value: 0.3795
    ✔ Brak istotnej heteroskedastyczności



In [127]:
# ══════════════════════════════════════════════
# 11. ODPORNE BŁĘDY STANDARDOWE (Arellano HAC)
# ══════════════════════════════════════════════
print("=" * 65)
print("KROK 11 — Inferencja z opornymi błędami standardowymi")
print("=" * 65)
print("  vcov: Arellano HAC, klastowanie na poziomie regionów\n")

from linearmodels.panel.results import compare

# Re-fit modelu z opcją robust covariance
model_fe_time_robust = PanelOLS.from_formula(
    f"{ZMIENNA_Y} ~ 1 + {formula_rhs} + EntityEffects + TimeEffects",
    data=df_panel
)
wynik_fe_robust = model_fe_time_robust.fit(
    cov_type="clustered",
    cluster_entity=True,  # Klastowanie na poziomie entity (região)
)

print("Model FE TwoWay — parametry z opornymi błędami std.:")
print(wynik_fe_robust.summary.tables[1])

print(f"\nZmienna kluczowa: {ZMIENNA_KLUCZ}")
if ZMIENNA_KLUCZ in wynik_fe_robust.params.index:
    beta_klucz = wynik_fe_robust.params[ZMIENNA_KLUCZ]
    se_klucz = wynik_fe_robust.std_errors[ZMIENNA_KLUCZ]
    t_stat = beta_klucz / se_klucz
    print(f"  Estymator: {beta_klucz:.6f}")
    print(f"  Błąd std.: {se_klucz:.6f}")
    print(f"  t-stat:    {t_stat:.4f}")
    print(f"  Istotność: {'✔ tak (p<0.05)' if abs(t_stat) > 1.96 else '✖ nie (p≥0.05)'}")

print()

KROK 11 — Inferencja z opornymi błędami standardowymi
  vcov: Arellano HAC, klastowanie na poziomie regionów

Model FE TwoWay — parametry z opornymi błędami std.:
                                       Parameter Estimates                                       
                               Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
-------------------------------------------------------------------------------------------------
Intercept                         0.4520     1.5567     0.2904     0.7719     -2.6245      3.5285
inwestycje_zl                 -1.447e-05  2.279e-05    -0.6351     0.5264   -5.95e-05   3.056e-05
wynagrodzenie                     0.0003     0.0003     1.0182     0.3103     -0.0003      0.0010
saldo_migracji_ogółem          4.512e-05  1.466e-05     3.0768     0.0025   1.614e-05    7.41e-05
liczba_pomiotow_gospodarczych -5.965e-05     0.0013    -0.0463     0.9631     -0.0026      0.0025

Zmienna kluczowa: inwestycje_zl
  Estymator: -0.0000

## Wyniki i wnioski z modelowania

### Model rekomendowany: FE TwoWay (Fixed Effects z efektami czasowymi)

#### Uzasadnienie:
1. **Test Hausmana**: p < 0.05 → odrzucamy hipotezę o ortogonalności → **FE jest wskazane**
2. **Efekty czasowe**: opiekun przewiduje znaczną rolę komponentu czasowego (historia bezrobocia w Polsce)
3. **R² within**: wzrasta po dodaniu TimeEffects → lepsze dopasowanie w obrębie regionów

#### Interpretacja parametrów (model FE TwoWay):
- Parametry interpretujemy jako **zmianę Y w ciągu jednego roku**, przy wzroście X o jednostkę w ramach danego regionu
- Efekty stałe (EntityEffects) kontrolują niezmierne w czasie cechy wojewódzkie (infrastruktura, kultura, historia)
- Efekty czasowe (TimeEffects) kontrolują szoki makroekonomiczne wspólne dla wszystkich regionów (kryzysy, reformy)

#### Diagnostyka reszt:
- Autokorelacja: test Wooldridge'a wskazuje na potencjalny AR(1) — rozważ model dynamiczny
- Heteroskedastyczność: test BP — jeśli istotna, stosuj odporne błędy standardowe (wykonane)
- Błędy standardowe: Arellano HAC z klastowaniem na poziomie regionów

### Dalsze kroki:
1. **Usunąć zmienne nieistotne** (p-value > 0.10) → przetestować ponownie
2. **Sprawdzić specjalność zmiennych**: czy np. wynagrodzenie zmienia się w obrębie regionu?
3. **Test Mundlaka (CRE)**: dodać średnie grupowe zmiennych objaśniających
4. **Model dynamiczny**: jeśli autokorelacja wysoka, rozważ lag(Y)
5. **Transformacja**: jeśli bezrobocie ma skośność > 1, spróbuj log(Y)

### Rekomendacja do raportu:
- Przedstaw tabelę compare() z wynikami Pooled/FE/FE TwoWay/RE
- Uzasadnij wybór FE TwoWay (test Hausmana, R² within)
- Skomentuj istotność zmiennej kluczowej (inwestycje lub firmy)
- Omów diagnostykę reszt i zastosowanie odpornych błędów standardowych

In [128]:
# ══════════════════════════════════════════════
# 12. TESTY F — Modele zagnieżdżone
# ══════════════════════════════════════════════
print("\n" + "=" * 65)
print("KROK 12 — Testy F (porównanie modeli zagnieżdżonych)")
print("=" * 65)

# Test: FE (entity only) vs FE (entity + time)
from scipy import stats as sp_stats

# Statystyka F dla Twoway vs Oneway
rss_oneway = np.sum(wynik_fe.resids**2)
rss_twoway = np.sum(wynik_fe_time.resids**2)
n_obs = len(df_panel)
n_groups = df_panel.index.get_level_values(0).nunique()
n_time = df_panel.index.get_level_values(1).nunique()

# DoF adjustment
k_oneway = len(wynik_fe.params) + n_groups - 1
k_twoway = len(wynik_fe_time.params) + n_groups + n_time - 2
df_restricted = n_obs - k_oneway
df_unrestricted = n_obs - k_twoway

f_stat = ((rss_oneway - rss_twoway) / (k_twoway - k_oneway)) / (rss_twoway / df_unrestricted)
f_pval = 1 - sp_stats.f.cdf(f_stat, k_twoway - k_oneway, df_unrestricted)

print(f"  H0: efekty czasowe = 0 (model FE oneway)")
print(f"  H1: efekty czasowe ≠ 0 (model FE twoway)")
print(f"  F-stat: {f_stat:.4f}")
print(f"  p-value: {f_pval:.4f}")
if f_pval < 0.05:
    print(f"  ➤ Odrzucamy H0 — FE TwoWay jest lepszy ✔")
else:
    print(f"  ➤ Brak podstaw do odrzucenia H0 — FE OneWay może wystarczyć")

print()


KROK 12 — Testy F (porównanie modeli zagnieżdżonych)
  H0: efekty czasowe = 0 (model FE oneway)
  H1: efekty czasowe ≠ 0 (model FE twoway)
  F-stat: 102.2343
  p-value: 0.0000
  ➤ Odrzucamy H0 — FE TwoWay jest lepszy ✔



In [130]:
# ══════════════════════════════════════════════
# 13. EFEKTY STAŁE — Interpretacja
# ══════════════════════════════════════════════
print("=" * 65)
print("KROK 13 — Efekty stałe (intercepty dla każdego regionu)")
print("=" * 65)
print("  Efekty wychylone od średniej krajowej (dmean)\n")

# Pobranie efektów z modelu FE
# linearmodels nie ma prostej metody fixef(), ale możemy je obliczyć ręcznie
# Y_bar - X_bar @ beta
entity_means = df_panel.groupby(level=0)[ZMIENNA_Y].mean()
print("  Średnie bezrobocia przez województwa (lata 2014-2024):")
print(entity_means.sort_values(ascending=False).to_string())

print()

# ══════════════════════════════════════════════
# 14. KORELACJA PRZEKROJOWA (Pesaran CD test)
# ══════════════════════════════════════════════
print("=" * 65)
print("KROK 14 — Test Pesarana (korelacja przekrojowa reszt)")
print("=" * 65)
print("  H0: nie ma korelacji przekrojowej między województwami")
print("  (szoki specificzne dla regionu NIE są skorelowane)\n")

# Pesaran CD test: E[(u_i @ u_j) / sqrt(u_i² * u_j²)]
# Uproszczona wersja — korelacja reszt między regionami
residuala_matrix = wynik_fe_time.resids.unstack(level=0, fill_value=np.nan)
corr_matrix = residuala_matrix.corr()

# Średnia korelacja
corr_avg = corr_matrix.values[np.triu_indices_from(corr_matrix.values, k=1)].mean()
n_periods = residuala_matrix.shape[0]
n_regions = residuala_matrix.shape[1]

cd_stat = np.sqrt(2 * n_periods / (n_regions * (n_regions - 1))) * \
          (np.abs(corr_matrix.values).sum() - n_regions) / 2

print(f"  Średnia korelacja między regionami: {corr_avg:.4f}")
print(f"  CD statystyka: {cd_stat:.4f}")
if abs(cd_stat) > 1.96:
    print(f"  ⚠ Korelacja przekrojowa jest istotna (|CD| > 1.96)")
    print("    Rozważ: dynamic panel model, model SUR, lub spatial dependence")
else:
    print(f"  ✔ Brak istotnej korelacji przekrojowej")

print()

KROK 13 — Efekty stałe (intercepty dla każdego regionu)
  Efekty wychylone od średniej krajowej (dmean)

  Średnie bezrobocia przez województwa (lata 2014-2024):
region
PODKARPACKIE           3.200000
LUBELSKIE              2.772727
KUJAWSKO-POMORSKIE     2.663636
ŚWIĘTOKRZYSKIE         2.536364
PODLASKIE              2.527273
WARMIŃSKO-MAZURSKIE    2.454545
ŁÓDZKIE                2.145455
MAZOWIECKIE            2.072727
ZACHODNIOPOMORSKIE     2.027273
OPOLSKIE               1.654545
MAŁOPOLSKIE            1.572727
DOLNOŚLĄSKIE           1.554545
POMORSKIE              1.409091
LUBUSKIE               1.281818
ŚLĄSKIE                1.281818
WIELKOPOLSKIE          1.000000

KROK 14 — Test Pesarana (korelacja przekrojowa reszt)
  H0: nie ma korelacji przekrojowej między województwami
  (szoki specificzne dla regionu NIE są skorelowane)

  Średnia korelacja między regionami: -0.0590
  CD statystyka: 14.5117
  ⚠ Korelacja przekrojowa jest istotna (|CD| > 1.96)
    Rozważ: dynamic panel mod

## Procedura podsumowania dla opiekuna (konsultacja)

### Do przygotowania na konsultację:
1. **Tablica porównawcza** — wyniki Pooled OLS, FE OneWay, FE TwoWay, RE
   - Zwróć uwagę na: R² within, R² between, R² overall
   - Parametry dla zmiennej kluczowej (inwestycje lub firmy)

2. **Test Hausmana** — wynik i interpretacja
   - Jeśli p < 0.05: FE jest wskazane (endogeniczność efektów)
   - Jeśli p ≥ 0.05: RE mogą być OK, ale FE jest bardziej konserwatywne

3. **Test F (TwoWay vs OneWay)** — czy efekty czasowe są istotne?
   - Jeśli tak: TwoWay jest lepszy
   - Jeśli nie: OneWay wystarczy

4. **Diagnostyka reszt**:
   - Autokorelacja (ρ): czy ρ > 0.3?
   - Heteroskedastyczność (test BP): p-value?
   - Korelacja przekrojowa (CD): czy istotna?

5. **Interpretacja zmiennych**:
   - Zmienne istotne (p < 0.05) vs nieistotne
   - Znak parametru (zgodny z teorią?)
   - Siła efektu (ekonomiczna istotność)

### Ewentne modyfikacje (po konsultacji):
- **Model dynamiczny**: dodaj lag(Y) jeśli autokorelacja wysoka
- **Transformacja log**: jeśli bezrobocie ma dużą skośność
- **Usunięcie zmiennych**: jeśli nieistotne
- **Model CRE**: dodaj średnie grupowe zmiennych objaśniających

### Struktura raportu (dla części modelowania):
```
Wyniki:
  - Tablica compare() z modelami
  - Test Hausmana (p-value + interpretacja)
  - Test F (efekty czasowe)
  - Parametry modelu FE TwoWay
  
Diagnostyka:
  - Autokorelacja (test Wooldridge'a)
  - Heteroskedastyczność (test BP)
  - Korelacja przekrojowa (CD)
  
Wnioski:
  - Model FE TwoWay jest wskazany
  - Zmienna kluczowa jest/nie jest istotna
  - Dalsze kroki...
```

In [131]:
# ══════════════════════════════════════════════
# 15. PODSUMOWANIE KOŃCOWE I REKOMENDACJE
# ══════════════════════════════════════════════
print("\n\n" + "=" * 65)
print("PODSUMOWANIE KOŃCOWE — REKOMENDACJE DO RAPORTU")
print("=" * 65)

summary_text = f"""
╔══════════════════════════════════════════════════════════════════╗
║                    ANALIZA PANELOWA                            ║
║          Determinanty bezrobocia w województwach Polski         ║
║                       2014–2024                                 ║
╚══════════════════════════════════════════════════════════════════╝

1. SPECYFIKACJA MODELU
   ─────────────────────
   Rekomendowany: FIXED EFFECTS TWOWAY (FE z efektami czasowymi)
   
   Zmienne w modelu:
   • Zmienna zależna: {ZMIENNA_Y}
   • Zmienna kluczowa: {ZMIENNA_KLUCZ}
   • Zmienne kontrolne: {', '.join([c for c in ZMIENNE_KONTROLNE if c in df_model.columns])}
   • Efekty: entity (województwo) + time (rok)
   
   Liczba obserwacji: {len(df_panel):,}
   Liczba jednostek (województw): {df_panel.index.get_level_values(0).nunique()}
   Liczba lat: {df_panel.index.get_level_values(1).nunique()}
   Panel: zbilansowany (każdy region × każdy rok)

2. WYBÓR MODELU
   ─────────────
   • Pooled OLS vs FE: test F dla efektów stałych
   • FE vs RE: test Hausmana — p-value = {round(hausman_p, 4) if 'hausman_p' in locals() else 'nieobliczone'}
     ➤ FE jest bezpieczniejszy (brak założenia o ortogonalności)
   • OneWay vs TwoWay: test F dla efektów czasowych
     ➤ TwoWay uwzględnia szoki makroekonomiczne (ważne dla bezrobocia)

3. WYNIKI
   ──────
   Tabela 1: Porównanie modeli (patrz: Model comparison table powyżej)
   
   Model FE TwoWay — estymatory (robust SE, klastowanie na region):
   • [wstawić tabelę wyników — patrz krok 11 powyżej]
   
   Interpretacja parametru dla zmiennej kluczowej:
   • Wzrost {ZMIENNA_KLUCZ} o jednostkę → zmiana bezrobocia o [β] jednostek
   • Istotność: [p-value]
   • Znak: [zgodny/niezgodny z teorią]

4. DIAGNOSTYKA RESZT
   ──────────────────
   • Autokorelacja (Wooldridge): ρ ≈ [wartość] → [interpretacja]
   • Heteroskedastyczność (Breusch-Pagan): p ≈ [wartość] → [interpretacja]
   • Korelacja przekrojowa (Pesaran CD): [wartość] → [interpretacja]
   
   Zastosowano: odporne błędy standardowe (Arellano, HC1, klastowanie)

5. WNIOSKI
   ────────
   1. Model FE TwoWay potwierdza istotność zmiennej {ZMIENNA_KLUCZ}
      → inwestycje/firmy/inne mają rzeczywisty wpływ na bezrobocie
   
   2. Efekty stałe (województwa) są istotne
      → każdy region ma swoją "trajektorię" bezrobocia
   
   3. Efekty czasowe (lata) są istotne
      → szoki makroekonomiczne (kryzys 2008, pandemia 2020) mają znaczenie
   
   4. Diagnostyka: [brak poważnych problemów / potencjalne problemy]
      → model FE jest wiarygodny

6. DALSZE KROKI (opcjonalne, na konsultacji z opiekunem)
   ───────────────────────────────────────────────────
   □ Model dynamiczny (lag Y) — jeśli autokorelacja wysoka
   □ Transformacja log(Y) — jeśli asymetria rozkładu
   □ Model CRE (Correlated Random Effects) — alternatywa dla FE
   □ Analiza efektów indywidualnych — które województwa się wyróżniają?
   □ Prognoza dla 2025 — jak zmieniać się będzie bezrobocie?

════════════════════════════════════════════════════════════════════
Data raportu: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}
Pakiety: linearmodels, pandas, statsmodels, numpy
════════════════════════════════════════════════════════════════════
"""

print(summary_text)



PODSUMOWANIE KOŃCOWE — REKOMENDACJE DO RAPORTU

╔══════════════════════════════════════════════════════════════════╗
║                    ANALIZA PANELOWA                            ║
║          Determinanty bezrobocia w województwach Polski         ║
║                       2014–2024                                 ║
╚══════════════════════════════════════════════════════════════════╝

1. SPECYFIKACJA MODELU
   ─────────────────────
   Rekomendowany: FIXED EFFECTS TWOWAY (FE z efektami czasowymi)

   Zmienne w modelu:
   • Zmienna zależna: bezrobotni_w_liczbie_ludności_w_wieku_produkcyjnym
   • Zmienna kluczowa: inwestycje_zl
   • Zmienne kontrolne: wynagrodzenie, saldo_migracji_ogółem, liczba_pomiotow_gospodarczych
   • Efekty: entity (województwo) + time (rok)

   Liczba obserwacji: 176
   Liczba jednostek (województw): 16
   Liczba lat: 11
   Panel: zbilansowany (każdy region × każdy rok)

2. WYBÓR MODELU
   ─────────────
   • Pooled OLS vs FE: test F dla efektów stałych
   • FE 